# Keyboard Model Runner (Colab)\n\nRuns your trained quadruped policy and lets you set commands from keyboard keys.\n\n- **W/S**: increase/decrease forward speed\n- **A/D**: increase/decrease lateral speed\n- **Q/E**: increase/decrease turn rate\n- **Space**: zero commands\n- **R**: run model (dry-run) with current command\n

In [ ]:
# If your repo is in Drive, mount first (optional)\nfrom google.colab import drive\ndrive.mount('/content/drive')

In [ ]:
# Set this to your project path in Colab\nPROJECT = '/content/drive/MyDrive/2026/robotics/quadruped'  # <-- edit if needed\n%cd {PROJECT}\n!pwd\n!ls

In [ ]:
# Install runtime deps for notebook control + inference\n!pip -q install onnxruntime numpy

In [ ]:
import os, subprocess, json\nfrom google.colab import output\nfrom IPython.display import display, HTML, Javascript\n\nstate = {\n    'speed': 0.0,     # vx\n    'lateral': 0.0,   # vy\n    'heading': 0.0,   # wz\n    'max_speed': 0.3,\n    'max_lateral': 0.2,\n    'max_heading': 0.5,\n    'step_v': 0.05,\n    'step_w': 0.10,\n}\n\ndef _clip(v, lo, hi):\n    return max(lo, min(hi, v))\n\ndef _status_html():\n    return f"""\n    <div style='font-family: monospace; font-size:16px; padding:10px; border:1px solid #ddd; border-radius:8px'>\n      <b>Current command</b><br/>\n      speed (vx):   {state['speed']:+.2f} m/s<br/>\n      lateral (vy): {state['lateral']:+.2f} m/s<br/>\n      heading (wz): {state['heading']:+.2f} rad/s\n    </div>\n    """\n\ndef refresh_status():\n    display(HTML(_status_html()))\n\ndef on_key(key):\n    k = (key or '').lower()\n    if k == 'w':\n        state['speed'] = _clip(state['speed'] + state['step_v'], -state['max_speed'], state['max_speed'])\n    elif k == 's':\n        state['speed'] = _clip(state['speed'] - state['step_v'], -state['max_speed'], state['max_speed'])\n    elif k == 'a':\n        state['lateral'] = _clip(state['lateral'] + state['step_v'], -state['max_lateral'], state['max_lateral'])\n    elif k == 'd':\n        state['lateral'] = _clip(state['lateral'] - state['step_v'], -state['max_lateral'], state['max_lateral'])\n    elif k == 'q':\n        state['heading'] = _clip(state['heading'] + state['step_w'], -state['max_heading'], state['max_heading'])\n    elif k == 'e':\n        state['heading'] = _clip(state['heading'] - state['step_w'], -state['max_heading'], state['max_heading'])\n    elif k == ' ':\n        state['speed'] = state['lateral'] = state['heading'] = 0.0\n    elif k == 'r':\n        run_model()\n    refresh_status()\n\ndef run_model(duration=10):\n    ckpt = 'deploy/policy.onnx' if os.path.exists('deploy/policy.onnx') else 'models/model_8000.pt'\n    cmd = [\n        'python', 'deploy/deploy.py',\n        '--checkpoint', ckpt,\n        '--dry-run',\n        '--duration', str(duration),\n        '--speed', str(state['speed']),\n        '--lateral', str(state['lateral']),\n        '--heading', str(state['heading']),\n    ]\n    print('Running:', ' '.join(cmd))\n    proc = subprocess.run(cmd, cwd=PROJECT, text=True, capture_output=True)\n    print(proc.stdout[-4000:])\n    if proc.returncode != 0:\n        print('\n[stderr]\n', proc.stderr[-4000:])\n\noutput.register_callback('notebook.keypress', on_key)\n\ndisplay(HTML("""\n<div tabindex='0' id='kbbox' style='padding:14px; border:2px dashed #999; border-radius:8px; outline:none;'>\n  <b>Click here, then use keyboard:</b><br/>\n  W/S speed, A/D lateral, Q/E turn, Space reset, R run model\n</div>\n"""))\n\ndisplay(Javascript("""\nconst box = document.getElementById('kbbox');\nbox.focus();\nbox.addEventListener('keydown', async (e) => {\n  e.preventDefault();\n  await google.colab.kernel.invokeFunction('notebook.keypress', [e.key], {});\n});\n"""))\n\nrefresh_status()

In [ ]:
# Optional: run once manually without keyboard\n# run_model(duration=15)